# GPT-2 Native Perplexity Zero-Shot CEFR Classification

This notebook reproduces the GPT-2 native perplexity zero-shot experiment
from the `minimal-cefr` repository.

**Pipeline:**
1. Extract aggregate perplexity features from pre-trained GPT-2
2. Convert to numeric feature matrices
3. Train Logistic Regression + XGBoost classifiers on EFCAMDAT-train
4. Predict on 3 test sets (EFCAMDAT-test, CELVA-SP, KUPA-KEYS)
5. Generate ranked results summary

**Runtime:** ~20 min on T4 GPU with default limits, ~3h on CPU.

**Setup:** Go to `Runtime > Change runtime type > T4 GPU` before running.

---
## 0. Install Dependencies

In [ ]:
!pip install transformers torch xgboost scikit-learn pandas tqdm -q

---
## 1. Configuration

Set row limits and other experiment parameters.
Change any limit to `None` to use the full dataset (needed for paper reproduction).

In [ ]:
from pathlib import Path
from datetime import datetime
import json

# ── Row limits ────────────────────────────────────────────────────────
# Default: full data. Set to a number for quick testing (e.g., 2000, 500)
LIMITS = {
    "norm-EFCAMDAT-train": None,   # Full: 80k rows (~3 min with batching)
    "norm-EFCAMDAT-test":  None,   # Full: 20k rows (~45 sec with batching)
    "norm-CELVA-SP":       None,   # 1742 rows - always full
    "norm-KUPA-KEYS":      None,   # 1006 rows - always full
}

# ── Dataset roles ─────────────────────────────────────────────────────
DATASETS = {
    "norm-EFCAMDAT-train": "train",
    "norm-EFCAMDAT-test":  "test",
    "norm-CELVA-SP":       "test",
    "norm-KUPA-KEYS":      "test",
}

# ── Constants ─────────────────────────────────────────────────────────
CLASSIFIERS = ["logistic", "xgboost"]
CEFR_COL = "cefr_level"
FEATURE_DIR_NAME = "gpt2_native"

# ── Batching Configuration ──────────────────────────────────────────────
# Use new batched extraction for 15-20x speedup
USE_BATCHED_EXTRACTION = True  # Set to False to use original pipeline

# Batch size: larger = faster but uses more VRAM
# T4 GPU (15GB): 16-32
# GPU (8GB): 4-8
# CPU: 1
BATCH_SIZE = 16

# PROCESSING_MODE (ADVANCED - leave as "auto" for smart default behavior)
# 
# DEFAULT BEHAVIOR ("auto"):
#   ✓ Checkpoint exists? → Resume from checkpoint automatically
#   ✓ No checkpoint? → Start from beginning
#   → This makes the notebook idempotent: just run it repeatedly!
#
# OVERRIDE OPTIONS (for special cases):
#   • "short_only" → Process only short texts, skip long texts
#   • "long_only" → Process only long texts, skip short texts
#   • "all" → Force full processing (skip resume)
#   • "resume" → Force resume (fail if no checkpoint)
#
PROCESSING_MODE = "auto"  # Smart default: auto-resume if checkpoint exists!

# Long-text strategy: how to handle texts exceeding model's max_length
# - "immediate_context" (DEFAULT): Minimal compute, each token from immediate context
# - "sliding_window": Non-overlapping windows, slower but alternative
# - "truncate": Fast mode, loses tail tokens
LONG_TEXT_STRATEGY = "immediate_context"

# Checkpoint directory for batch recovery (persistent across notebook restarts)
CHECKPOINT_BASE = Path("/content/checkpoints")

# Skip merge: set to True to process batches but keep separate for later merge
SKIP_MERGE = False

# ── Paths ─────────────────────────────────────────────────────────────
# Data source (Google Drive)
GDRIVE_DATA_ROOT = Path('/content/drive/MyDrive/phd-experimental-data/cefr-classification')
DATA_PATH = GDRIVE_DATA_ROOT / 'data' / 'splits'

# Experiment output (local)
TIMESTAMP = datetime.now().strftime('%Y%m%d-%H%M%S')
EXP_NAME = f"experiment-gpt2-native-{TIMESTAMP}"
EXP_DIR = Path(f"/content/{EXP_NAME}")
PERPLEXITY_RAW = EXP_DIR / "perplexity-raw"
TRIMMED_LABELS = EXP_DIR / "trimmed-labels"

# Repo path (local)
REPO_DIR = Path("/content/minimal-cefr")

print("Configuration loaded.")
print(f"  Experiment: {EXP_NAME}")
print(f"  Extraction: {'BATCHED (2-3 min/80k)' if USE_BATCHED_EXTRACTION else 'STANDARD (30 min/80k)'}")
print(f"  Processing mode: {PROCESSING_MODE} (auto-resume if checkpoint exists)")
print(f"  Long-text strategy: {LONG_TEXT_STRATEGY}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Data: {DATA_PATH}")
print(f"  Checkpoints: {CHECKPOINT_BASE}")
print()
print("ℹ️  DEFAULT BEHAVIOR: Notebook will auto-resume from checkpoint if it exists.")
print("   Just re-run the notebook after any interruption - it picks up where it stopped!")


---
## 2. Data Acquisition

Run **ONE** of the two options below (A or B), then continue.

### Option A: Google Drive

Use this if your data CSVs are in Google Drive.
Expected location: `MyDrive/phd-experimental-data/cefr-classification/data/splits/` containing:
- `norm-EFCAMDAT-train.csv`
- `norm-EFCAMDAT-test.csv`
- `norm-CELVA-SP.csv`
- `norm-KUPA-KEYS.csv`

In [ ]:
# ── OPTION A: Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Full paths for Colab (synced from i:/phd-experimental-data/cefr-classification/)
GDRIVE_DATA_ROOT = Path('/content/drive/MyDrive/phd-experimental-data/cefr-classification')
DATA_PATH = GDRIVE_DATA_ROOT / 'data' / 'splits'

print(f"DATA_PATH = {DATA_PATH}")

### Option B: Download via URL

Use this if you have a direct download link to a .zip file
containing the data CSVs.

In [ ]:
# ── OPTION B: wget + unzip ───────────────────────────────────────────
# Replace the URL below with your actual download link
DATA_URL = "YOUR_URL_HERE"  # <-- paste your .zip URL here

!wget -q --show-progress -O /content/cefr-data.zip "{DATA_URL}"
!unzip -q -o /content/cefr-data.zip -d /content/data

# Adjust this path to match the structure inside your zip
DATA_PATH = Path('/content/data/splits')

# If your zip extracts differently, list the contents to find the right path:
# !find /content/data -name '*.csv' | head -10

print(f"DATA_PATH = {DATA_PATH}")

### Verify Data

Check that all required files are present.

In [ ]:
assert DATA_PATH is not None, "Run Option A or Option B above first!"

missing = []
for name in DATASETS:
    f = DATA_PATH / f"{name}.csv"
    if f.exists():
        import pandas as pd
        n = len(pd.read_csv(f))
        print(f"  OK  {name}.csv ({n:,} rows)")
    else:
        print(f"  MISSING  {f}")
        missing.append(name)

if missing:
    raise FileNotFoundError(
        f"Missing files: {missing}. Check DATA_PATH or re-run data acquisition."
    )

---
## 3. Get Repository Code

Clone the `minimal-cefr` repository to get the `src/` modules.

If the repo is private, use **Alternative** below instead.

In [ ]:
# Clone the repository (change URL to your fork if needed)
!git clone https://github.com/YOUR_USERNAME/minimal-cefr.git /content/minimal-cefr 2>/dev/null || echo "Repository already cloned."

%cd /content/minimal-cefr

# Verify src/ exists
!ls src/*.py | head -5

**Alternative:** If you don't have a public repo, upload a zip of the `src/` directory:

```python
from google.colab import files
uploaded = files.upload()  # upload src.zip
!mkdir -p /content/minimal-cefr
!unzip -q -o src.zip -d /content/minimal-cefr
%cd /content/minimal-cefr
```

---
## 4. Experiment Setup

Create directory structure, copy data, detect GPU.

In [ ]:
import shutil
import torch

# ── Device detection ──────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("  WARNING: No GPU detected. Perplexity extraction will be slow.")
    print("  Go to Runtime > Change runtime type > T4 GPU")

# ── Create experiment directories ─────────────────────────────────────
for d in [
    EXP_DIR / "ml-training-data",
    EXP_DIR / "ml-test-data",
    EXP_DIR / "features",
    EXP_DIR / "feature-models",
    EXP_DIR / "results",
    PERPLEXITY_RAW,
    TRIMMED_LABELS,
]:
    d.mkdir(parents=True, exist_ok=True)

# ── Copy data into experiment structure ───────────────────────────────
for name, role in DATASETS.items():
    src = DATA_PATH / f"{name}.csv"
    if role == "train":
        dest = EXP_DIR / "ml-training-data" / f"{name}.csv"
    else:
        dest = EXP_DIR / "ml-test-data" / f"{name}.csv"

    if not dest.exists():
        shutil.copy2(src, dest)
        print(f"  Copied {name}.csv -> {dest.parent.name}/")
    else:
        print(f"  Already exists: {dest.name}")

print("\nSetup complete.")

In [ ]:
### Smart Default: Auto-Resume from Checkpoint

**How it works:**
1. Run the notebook normally
2. If interrupted → Colab session ends
3. Open the notebook again and run it → **Automatically resumes from checkpoint!**
4. No configuration needed - it just works!

This is the perfect workflow for Google Drive-backed experiments:
- Checkpoints persist in Google Drive
- Just keep re-running the notebook
- It picks up exactly where it stopped

**Example workflow:**

**Session 1:** `PROCESSING_MODE = "auto"` (default)
```
✓ Process dataset 1 (short texts): 2 min
✓ Process dataset 1 (long texts): 45 min
→ INTERRUPTED after 47 minutes
Checkpoint saved to Google Drive
```

**Session 2:** Same notebook, re-run it
```
Checking checkpoint...
  Checkpoint found! Auto-resuming...
    Short texts: 3500/3500 batches already done → Skipping
    Long texts: 18000/24000 batches
  Mode: RESUME from checkpoint
✓ Continue from batch 18000...
✓ Finish remaining 6000 long batches (15 more minutes)
✓ Move to next datasets
```

**No configuration needed!**

---

### ADVANCED: Processing Mode Overrides

If you need special behavior, change `PROCESSING_MODE`:

| Mode | Behavior | Use Case |
|------|----------|----------|
| `"auto"` (DEFAULT) | Resume if checkpoint exists, else start fresh | Normal usage - just works! |
| `"resume"` | Force resume (fail if no checkpoint) | Explicitly resume |
| `"all"` | Force full processing (ignore checkpoint) | Re-process everything |
| `"short_only"` | Process only short texts | Testing, or spread across days |
| `"long_only"` | Process only long texts | After short texts are done |

**Most of the time, just leave `PROCESSING_MODE = "auto"`**


### Quick Reference: Processing Modes

**How to process datasets incrementally (slowly over time):**

1. **First run (everything):**
   ```
   PROCESSING_MODE = "all"
   SKIP_MERGE = False
   ```
   → Processes short texts, long texts, and merges all results

2. **Process short texts only, long texts later:**
   ```
   PROCESSING_MODE = "short_only"
   SKIP_MERGE = True
   ```
   → Run this first, then come back later for long texts

3. **Process only long texts (after short are done):**
   ```
   PROCESSING_MODE = "long_only"
   SKIP_MERGE = True
   ```
   → Run after short texts are done

4. **Resume if interrupted:**
   ```
   PROCESSING_MODE = "resume"
   SKIP_MERGE = False
   ```
   → Automatically continues from last checkpoint

5. **Manually merge batches later:**
   ```
   SKIP_MERGE = True  # On initial run
   ```
   → Then run cell 5a (Manual Batch Merge) when ready

**Processing times:**
- Short texts (56k): ~2 min on T4 GPU
- Long texts (24k): ~30-60 min on T4 GPU (depending on text lengths)
- Total: ~45 min to 1 hour on T4 GPU

**GPU memory usage:**
- BATCH_SIZE=16: ~12-14 GB on T4
- BATCH_SIZE=8: ~7-8 GB (for smaller GPUs)

%%time
import subprocess
import sys
import pandas as pd

# Verify setup
if not REPO_DIR.exists():
    raise RuntimeError(
        f"Repository not found at {REPO_DIR}. "
        "Run cell 13 (Get Repository Code) first!"
    )

if not DATA_PATH.exists():
    raise RuntimeError(
        f"Data not found at {DATA_PATH}. "
        "Run cell 7 or 8 (Data Acquisition) first!"
    )

# ── BATCHED EXTRACTION WITH AUTO-RESUME ──────────────────────────────
if USE_BATCHED_EXTRACTION:
    for name in DATASETS:
        limit = LIMITS.get(name)
        src_csv = DATA_PATH / f"{name}.csv"
        checkpoint_dir = CHECKPOINT_BASE / f"{name}-{TIMESTAMP}"
        raw_out = PERPLEXITY_RAW / f"{name}.csv"
        
        print(f"\n{'='*70}")
        print(f"DATASET: {name}")
        print(f"{'='*70}")
        print(f"  Limit: {limit if limit else 'FULL'}")
        
        # Check if already done
        if raw_out.exists():
            print(f"  ✓ Perplexity already extracted: {raw_out.name}")
            # Convert to features only
        else:
            # Build extraction command
            cmd = [
                sys.executable, "-m", "src.extract_perplexity_features_batched",
                "-i", str(src_csv),
                "--text-column", "text",
                "-m", "gpt2",
                "-d", DEVICE,
                "--aggregate-only",
                "--batch-size", str(BATCH_SIZE),
                "--checkpoint-dir", str(checkpoint_dir),
                "--long-text-strategy", LONG_TEXT_STRATEGY,
                "-f", "csv",
                "-o", str(raw_out),
            ]
            
            # ── SMART AUTO-RESUME LOGIC ──────────────────────────────────────
            # Check if checkpoint already exists
            checkpoint_metadata = checkpoint_dir / "metadata.json"
            
            if checkpoint_metadata.exists():
                # Checkpoint exists → Auto-resume!
                with open(checkpoint_metadata) as f:
                    checkpoint_state = json.load(f)
                
                short_done = checkpoint_state.get("short_texts", {}).get("completed_batches", 0)
                long_done = checkpoint_state.get("long_texts", {}).get("completed_batches", 0)
                
                print(f"  Checkpoint found! Auto-resuming...")
                print(f"    Short texts: {short_done} batches already done")
                print(f"    Long texts: {long_done} batches already done")
                
                # Determine what to resume
                short_total = checkpoint_state.get("short_texts", {}).get("total_batches", 0)
                long_total = checkpoint_state.get("long_texts", {}).get("total_batches", 0)
                
                # Smart skip logic: if short done, skip short texts
                if short_done >= short_total and short_total > 0:
                    print(f"    → Short texts already 100% done, skipping")
                    cmd.append("--skip-short-texts")
                
                # Add resume flag
                cmd.append("--resume")
                print(f"  Mode: RESUME from checkpoint")
            else:
                # No checkpoint → Start from beginning
                print(f"  Checkpoint: Starting fresh (no checkpoint yet)")
                print(f"  Mode: START FROM BEGINNING")
            
            # Handle PROCESSING_MODE overrides for special cases
            if PROCESSING_MODE == "short_only":
                cmd.append("--skip-long-texts")
                print(f"  Override: SHORT TEXTS ONLY")
            elif PROCESSING_MODE == "long_only":
                cmd.append("--skip-short-texts")
                print(f"  Override: LONG TEXTS ONLY")
            elif PROCESSING_MODE == "all":
                # Remove --resume if user forces "all" mode
                if "--resume" in cmd:
                    cmd.remove("--resume")
                print(f"  Override: FORCE FULL PROCESSING (ignoring checkpoint)")
            # "auto" and "resume" modes use default logic above
            
            if SKIP_MERGE:
                cmd.append("--no-merge")
                print(f"  Merge: DISABLED (batches kept separate)")
            
            if limit is not None:
                cmd.extend(["--limit", str(limit)])
            
            print(f"  Strategy: {LONG_TEXT_STRATEGY}")
            print(f"  Batch size: {BATCH_SIZE}")
            print(f"\n  Starting extraction...")
            
            result = subprocess.run(
                cmd, 
                cwd=str(REPO_DIR), 
                capture_output=True, 
                text=True, 
                timeout=7200
            )
            
            if result.returncode != 0:
                print(f"  ❌ FAILED")
                print(f"  STDERR: {result.stderr[-2000:]}")
                raise RuntimeError(f"Batched extraction failed for {name}")
            
            # Show last few lines of output
            for line in result.stdout.strip().split('\n')[-8:]:
                if line.strip():
                    print(f"  {line}")
        
        # Step 2: Convert to features
        feat_dir = EXP_DIR / "features" / FEATURE_DIR_NAME / name
        feat_dir.mkdir(parents=True, exist_ok=True)
        feat_out = feat_dir / "features_dense.csv"
        
        if not feat_out.exists():
            print(f"\n  Converting to feature matrix...")
            df = pd.read_csv(raw_out)
            drop_cols = [c for c in ["text", "model"] if c in df.columns]
            df_numeric = df.drop(columns=drop_cols)
            
            for col in df_numeric.columns:
                df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")
            
            df_numeric = df_numeric.fillna(0.0)
            df_numeric.to_csv(feat_out, index=False)
            
            fn_file = feat_dir / "feature_names.csv"
            pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)
            
            print(f"  ✓ Features: {len(df_numeric):,} rows × {len(df_numeric.columns)} cols")
            print(f"    Saved: {feat_out}")
        else:
            print(f"  ✓ Features already converted: {feat_out.name}")

# ── ORIGINAL EXTRACTION (fallback) ──────────────────────────────────────
else:
    print("\n⚠️  Using standard (non-batched) extraction - will be slower")
    print("   Set USE_BATCHED_EXTRACTION = True for 10-15x speedup\n")
    
    for name in DATASETS:
        limit = LIMITS.get(name)
        src_csv = DATA_PATH / f"{name}.csv"
        raw_out = PERPLEXITY_RAW / f"{name}.csv"
        
        print(f"\n{'='*70}")
        print(f"  {name} (limit={limit})")
        print(f"{'='*70}")
        
        if not raw_out.exists():
            cmd = [
                sys.executable, "-m", "src.extract_perplexity_features",
                "-i", str(src_csv),
                "--text-column", "text",
                "-m", "gpt2",
                "-d", DEVICE,
                "--aggregate-only",
                "-f", "csv",
                "-o", str(raw_out),
            ]
            if limit is not None:
                cmd.extend(["--limit", str(limit)])
            
            print(f"  Extracting perplexity...")
            result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True, timeout=7200)
            if result.returncode != 0:
                print(f"  STDERR: {result.stderr[-2000:]}")
                raise RuntimeError(f"Perplexity extraction failed for {name}")
            
            for line in result.stdout.strip().split('\n')[-5:]:
                print(f"  {line}")
        else:
            print(f"  ✓ Perplexity already extracted.")
        
        # Convert to features
        feat_dir = EXP_DIR / "features" / FEATURE_DIR_NAME / name
        feat_dir.mkdir(parents=True, exist_ok=True)
        feat_out = feat_dir / "features_dense.csv"
        
        if not feat_out.exists():
            df = pd.read_csv(raw_out)
            drop_cols = [c for c in ["text", "model"] if c in df.columns]
            df_numeric = df.drop(columns=drop_cols)
            
            for col in df_numeric.columns:
                df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")
            
            df_numeric = df_numeric.fillna(0.0)
            df_numeric.to_csv(feat_out, index=False)
            
            fn_file = feat_dir / "feature_names.csv"
            pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)
            
            print(f"  ✓ Features: {len(df_numeric)} rows x {len(df_numeric.columns)} cols")
        else:
            print(f"  ✓ Features already converted.")

print("\n" + "="*70)
print("Step 1+2 complete.")
print("="*70)


In [ ]:
# Manually merge batches for datasets that need it
import subprocess
import sys

print("MANUAL BATCH MERGE")
print("=" * 70)

for name in DATASETS:
    checkpoint_dir = CHECKPOINT_BASE / f"{name}-{TIMESTAMP}"
    raw_out = PERPLEXITY_RAW / f"{name}.csv"
    
    metadata_file = checkpoint_dir / "metadata.json"
    
    if not metadata_file.exists():
        print(f"\n{name}: No checkpoint found, skipping")
        continue
    
    with open(metadata_file) as f:
        metadata = json.load(f)
    
    # Skip if already merged
    if metadata.get("merged"):
        print(f"\n{name}: ✓ Already merged")
        continue
    
    print(f"\n{name}")
    print(f"  Merging batches...")
    
    cmd = [
        sys.executable, "-m", "scripts.merge_batch_outputs",
        "--checkpoint-dir", str(checkpoint_dir),
        "-o", str(raw_out),
        "-f", "csv",
    ]
    
    result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    
    if result.returncode != 0:
        print(f"  ❌ FAILED: {result.stderr[-500:]}")
        raise RuntimeError(f"Merge failed for {name}")
    
    # Show output
    for line in result.stdout.strip().split('\n')[-5:]:
        if line.strip():
            print(f"  {line}")

print("\n" + "=" * 70)

---
## 5a. (Optional) Manually Merge Batches After Extraction

If you used `SKIP_MERGE = True` or `PROCESSING_MODE = "short_only"/"long_only"`,
run this cell to merge the batch results after processing is complete.

---
## 5. Step 1+2: Extract GPT-2 Perplexity & Convert to Features

For each dataset:
1. Run GPT-2 perplexity extraction (GPU-accelerated)
2. Convert raw output to numeric feature matrix

**Runtime:** ~15 min on T4 GPU with default limits, ~2h on CPU.

In [ ]:
%%timeimport subprocessimport sysimport pandas as pd# Verify setupif not REPO_DIR.exists():    raise RuntimeError(        f"Repository not found at {REPO_DIR}. "        "Run cell 13 (Get Repository Code) first!"    )if not DATA_PATH.exists():    raise RuntimeError(        f"Data not found at {DATA_PATH}. "        "Run cell 7 or 8 (Data Acquisition) first!"    )# ── BATCHED EXTRACTION ──────────────────────────────────────────────────if USE_BATCHED_EXTRACTION:    for name in DATASETS:        limit = LIMITS.get(name)        src_csv = DATA_PATH / f"{name}.csv"        checkpoint_dir = CHECKPOINT_BASE / f"{name}-{TIMESTAMP}"        raw_out = PERPLEXITY_RAW / f"{name}.csv"                print(f"\n{'='*60}")        print(f"  {name} (BATCHED)  (limit={limit})")        print(f"{'='*60}")                # Check if already done        if raw_out.exists():            print(f"  Perplexity already extracted: {raw_out.name}")        else:            # Use batched extraction            cmd = [                sys.executable, "-m", "src.extract_perplexity_features_batched",                "-i", str(src_csv),                "--text-column", "text",                "-m", "gpt2",                "-d", DEVICE,                "--aggregate-only",                "--batch-size", str(BATCH_SIZE),                "--checkpoint-dir", str(checkpoint_dir),                "-f", "csv",                "-o", str(raw_out),            ]            if limit is not None:                cmd.extend(["--limit", str(limit)])                        print(f"  Extracting with batch size {BATCH_SIZE}...")            result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True, timeout=7200)            if result.returncode != 0:                print(f"  STDERR: {result.stderr[-2000:]}")                raise RuntimeError(f"Batched extraction failed for {name}")                        for line in result.stdout.strip().split('\n')[-5:]:                print(f"  {line}")                # Step 2: Convert to features        feat_dir = EXP_DIR / "features" / FEATURE_DIR_NAME / name        feat_dir.mkdir(parents=True, exist_ok=True)        feat_out = feat_dir / "features_dense.csv"                if not feat_out.exists():            df = pd.read_csv(raw_out)            drop_cols = [c for c in ["text", "model"] if c in df.columns]            df_numeric = df.drop(columns=drop_cols)            for col in df_numeric.columns:                df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")            df_numeric = df_numeric.fillna(0.0)            df_numeric.to_csv(feat_out, index=False)                        fn_file = feat_dir / "feature_names.csv"            pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)                        print(f"  Features: {len(df_numeric)} rows x {len(df_numeric.columns)} cols")        else:            print(f"  Features already converted.")# ── ORIGINAL EXTRACTION (fallback) ──────────────────────────────────────else:    for name in DATASETS:        limit = LIMITS.get(name)        src_csv = DATA_PATH / f"{name}.csv"        raw_out = PERPLEXITY_RAW / f"{name}.csv"                print(f"\n{'='*60}")        print(f"  {name}  (limit={limit})")        print(f"{'='*60}")                if not raw_out.exists():            cmd = [                sys.executable, "-m", "src.extract_perplexity_features",                "-i", str(src_csv),                "--text-column", "text",                "-m", "gpt2",                "-d", DEVICE,                "--aggregate-only",                "-f", "csv",                "-o", str(raw_out),            ]            if limit is not None:                cmd.extend(["--limit", str(limit)])                        print(f"  Extracting perplexity...")            result = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True, timeout=7200)            if result.returncode != 0:                print(f"  STDERR: {result.stderr[-2000:]}")                raise RuntimeError(f"Perplexity extraction failed for {name}")                        for line in result.stdout.strip().split('\n')[-5:]:                print(f"  {line}")        else:            print(f"  Perplexity already extracted.")                # Convert to features        feat_dir = EXP_DIR / "features" / FEATURE_DIR_NAME / name        feat_dir.mkdir(parents=True, exist_ok=True)        feat_out = feat_dir / "features_dense.csv"                if not feat_out.exists():            df = pd.read_csv(raw_out)            drop_cols = [c for c in ["text", "model"] if c in df.columns]            df_numeric = df.drop(columns=drop_cols)            for col in df_numeric.columns:                df_numeric[col] = pd.to_numeric(df_numeric[col], errors="coerce")            df_numeric = df_numeric.fillna(0.0)            df_numeric.to_csv(feat_out, index=False)                        fn_file = feat_dir / "feature_names.csv"            pd.DataFrame({"feature_name": df_numeric.columns}).to_csv(fn_file, index=False)                        print(f"  Features: {len(df_numeric)} rows x {len(df_numeric.columns)} cols")        else:            print(f"  Features already converted.")print("\nStep 1+2 complete.")

---
## 6. Step 3: Train Classifiers (LR + XGBoost)

Train on EFCAMDAT-train perplexity features.

**Runtime:** ~10 seconds.

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

model_names = {}

features_file = (
    EXP_DIR / "features" / FEATURE_DIR_NAME / "norm-EFCAMDAT-train" / "features_dense.csv"
)
labels_csv = EXP_DIR / "ml-training-data" / "norm-EFCAMDAT-train.csv"

# Check if labels need trimming (due to --limit)
n_features = len(pd.read_csv(features_file))
df_labels = pd.read_csv(labels_csv)
if len(df_labels) != n_features:
    print(f"  Trimming labels from {len(df_labels)} to {n_features} rows")
    trimmed = TRIMMED_LABELS / "norm-EFCAMDAT-train.csv"
    df_labels.head(n_features).to_csv(trimmed, index=False)
    labels_csv = trimmed

for clf_type in CLASSIFIERS:
    model_name = f"norm-EFCAMDAT-train_{clf_type}_gpt2native"
    model_names[clf_type] = model_name
    model_dir = EXP_DIR / "feature-models" / "classifiers" / model_name

    if (model_dir / "classifier.pkl").exists():
        print(f"  Already trained: {model_name}")
        continue

    print(f"\n  Training {clf_type}...")
    cmd = [
        sys.executable, "-m", "src.train_classifiers",
        "-e", str(EXP_DIR),
        "--features-file", str(features_file),
        "--labels-csv", str(labels_csv),
        "--cefr-column", CEFR_COL,
        "--classifier", clf_type,
        "--model-name", model_name,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_DIR))
    if result.returncode != 0:
        print(f"  STDERR: {result.stderr[-2000:]}")
        raise RuntimeError(f"Training failed for {clf_type}")
    for line in result.stdout.strip().split('\n')[-5:]:
        print(f"  {line}")

print(f"\nTrained models: {list(model_names.values())}")

---
## 7. Step 4: Predict on Test Sets

Run each classifier on all 3 test sets (6 predictions total).

**Runtime:** ~10 seconds.

In [ ]:
%%time
import subprocess
import sys
import pandas as pd

test_sets = [n for n, role in DATASETS.items() if role == "test"]

for clf_type in CLASSIFIERS:
    model_name = model_names[clf_type]
    for test_name in test_sets:
        results_dir = EXP_DIR / "results" / model_name / test_name

        if (results_dir / "evaluation_report.md").exists():
            print(f"  Already done: {clf_type} -> {test_name}")
            continue

        features_file = (
            EXP_DIR / "features" / FEATURE_DIR_NAME / test_name / "features_dense.csv"
        )
        labels_csv = EXP_DIR / "ml-test-data" / f"{test_name}.csv"

        # Check if labels need trimming
        n_features = len(pd.read_csv(features_file))
        df_labels = pd.read_csv(labels_csv)
        if len(df_labels) != n_features:
            print(f"  Trimming {test_name} labels: {len(df_labels)} -> {n_features}")
            trimmed = TRIMMED_LABELS / f"{test_name}.csv"
            df_labels.head(n_features).to_csv(trimmed, index=False)
            labels_csv = trimmed

        print(f"  Predicting: {clf_type} -> {test_name}")
        cmd = [
            sys.executable, "-m", "src.predict",
            "-e", str(EXP_DIR),
            "-m", model_name,
            "--features-file", str(features_file),
            "--labels-csv", str(labels_csv),
            "--cefr-column", CEFR_COL,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_DIR))
        if result.returncode != 0:
            print(f"  STDERR: {result.stderr[-2000:]}")
            raise RuntimeError(f"Prediction failed: {model_name} -> {test_name}")
        for line in result.stdout.strip().split('\n')[-3:]:
            print(f"    {line}")

print("\nAll predictions complete.")

---
## 8. Step 5: Generate Report

Aggregate all evaluation results into a ranked summary table.

In [ ]:
import subprocess
import sys

summary_path = EXP_DIR / "results_summary.md"

cmd = [
    sys.executable, "-m", "src.report",
    "-e", str(EXP_DIR),
    "--rank", "accuracy",
    "--summary-report", str(summary_path),
    "--include-all-datasets",
    "-v",
]
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REPO_DIR))
if result.returncode != 0:
    print(f"STDERR: {result.stderr[-2000:]}")
    raise RuntimeError("Report generation failed")
print(result.stdout[-1000:])

---
## 9. Results

Display the results summary.

In [ ]:
from IPython.display import Markdown, display

summary_path = EXP_DIR / "results_summary.md"

if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    print("No results summary found. Run Step 5 above.")

### Individual Evaluation Reports

View detailed per-model, per-dataset evaluation reports.

In [ ]:
from IPython.display import Markdown, display

for clf_type in CLASSIFIERS:
    model_name = model_names[clf_type]
    test_sets = [n for n, role in DATASETS.items() if role == "test"]
    for test_name in test_sets:
        report_path = EXP_DIR / "results" / model_name / test_name / "evaluation_report.md"
        if report_path.exists():
            display(Markdown(f"---\n### {clf_type} on {test_name}"))
            display(Markdown(report_path.read_text()))
        else:
            print(f"  Missing: {report_path}")

---
## 10. Download Results (Optional)

Download the experiment outputs to your local machine.

In [ ]:
# Zip the experiment directory for download
!cd /content && zip -r -q experiment-results.zip experiment/results/ experiment/results_summary.md

from google.colab import files
files.download('/content/experiment-results.zip')